# Multi-Sport Organization — Fan Intelligence Pipeline

**Goal:** Pull ticket and merch data from a cloud data warehouse, clean both datasets,
join them at the fan level, and export a single master "fan superprofile" CSV.

> **Portfolio note:** Organization names, credentials, and table references have been
> fully abstracted. Set `DATA_SOURCE = 'csv'` (the default) to run end-to-end against
> the included synthetic sample data — no credentials required. Switch to `'snowflake'`
> and supply env vars to connect to your own instance.

### Pipeline Steps
1. Install dependencies & connect to data source
2. Pull raw data (two ticket sources + merch + ZIP reference)
3. Audit data quality (null rates, unique counts)
4. Clean tickets — drop junk columns, heal geography, normalize financials
5. Clean merch — drop junk columns, parse mixed `Customer Name` field, normalize financials
6. Build join keys on both sides (email tier → name fallback tier)
7. Aggregate tickets → one row per fan
8. Aggregate merch → one row per fan
9. Two-pass join: ticket fans ↔ merch fans
10. Sanity check
11. Export


---
## BLOCK 1 — Dependencies & Data Source Connection


In [ ]:
!pip install "snowflake-connector-python[pandas]" rapidfuzz --quiet

import os
import re
import warnings
import pandas as pd
from rapidfuzz import process

# ── Data source toggle ──────────────────────────────────────────────────────
# 'csv'       → load synthetic sample data from CSV_DIR (no credentials needed)
# 'snowflake' → connect to a live Snowflake instance via environment variables
DATA_SOURCE = os.environ.get('DATA_SOURCE', 'csv')
CSV_DIR     = os.environ.get('CSV_DIR', 'sample_data')

if DATA_SOURCE == 'snowflake':
    import snowflake.connector

    # Set these in your environment — never hardcode credentials in source:
    #   SF_USER, SF_PASSWORD, SF_ACCOUNT, SF_WAREHOUSE (optional), SF_ROLE (optional)
    conn = snowflake.connector.connect(
        user      = os.environ['SF_USER'],
        password  = os.environ['SF_PASSWORD'],
        account   = os.environ['SF_ACCOUNT'],
        warehouse = os.environ.get('SF_WAREHOUSE', 'compute_wh'),
        role      = os.environ.get('SF_ROLE', 'SYSADMIN'),
    )
    print("Connected to Snowflake.")
else:
    print(f"Running in CSV mode — loading from '{CSV_DIR}/'")


---
## BLOCK 2 — Pull Raw Data


In [ ]:
warnings.filterwarnings('ignore')  # suppress SQLAlchemy connector warning

if DATA_SOURCE == 'snowflake':
    # Database names are read from env vars so this notebook stays credential-free.
    # Override any of these to match your Snowflake schema.
    TEAM_A_DB = os.environ.get('SF_TEAM_A_DB',  'team_a_tickets')
    TEAM_B_DB = os.environ.get('SF_TEAM_B_DB',  'team_b_tickets')
    MERCH_DB  = os.environ.get('SF_MERCH_DB',   'merch')
    ZIP_DB    = os.environ.get('SF_ZIP_DB',      'us_zip_code')

    print("Pulling Team A ticket data...")
    df_tickets = pd.read_sql(f"SELECT * FROM {TEAM_A_DB}.public.raw_tickets", conn)
    print(f"  Tickets: {len(df_tickets):,} rows, {len(df_tickets.columns)} columns")

    print("Pulling merch data...")
    df_merch = pd.read_sql(f"SELECT * FROM {MERCH_DB}.public.square_transactions", conn)
    print(f"  Merch:   {len(df_merch):,} rows, {len(df_merch.columns)} columns")

    print("Pulling ZIP reference table...")
    df_ref = pd.read_sql(f"""
        SELECT DISTINCT ZIP, USPS_ZIP_PREF_CITY, USPS_ZIP_PREF_STATE
        FROM {ZIP_DB}.zipcode_crosswalk.zip_county
    """, conn)
    print(f"  ZIP ref: {len(df_ref):,} rows")

    print("Pulling Team B ticket data...")
    df_team_b = pd.read_sql(f"SELECT * FROM {TEAM_B_DB}.public.raw_tickets", conn)
    print(f"  Team B:  {len(df_team_b):,} rows, {len(df_team_b.columns)} columns")

else:
    print("Loading CSV sample data...")
    df_tickets  = pd.read_csv(f'{CSV_DIR}/raw_tickets_team_a.csv')
    df_merch    = pd.read_csv(f'{CSV_DIR}/raw_merch.csv')
    df_ref      = pd.read_csv(f'{CSV_DIR}/zip_reference.csv', dtype={'ZIP': str})
    df_team_b = pd.read_csv(f'{CSV_DIR}/raw_tickets_team_b.csv')
    print(f"  Team A tickets: {len(df_tickets):,} rows, {len(df_tickets.columns)} columns")
    print(f"  Merch:          {len(df_merch):,} rows, {len(df_merch.columns)} columns")
    print(f"  ZIP ref:        {len(df_ref):,} rows")
    print(f"  Team B tickets: {len(df_team_b):,} rows, {len(df_team_b.columns)} columns")

# ── Tag sport type before stacking ─────────────────────────────────────────
df_team_b.loc[
    df_team_b['HOME_TEAM'].str.strip().str.lower() == 'team b',
    'SPORT'
] = 'sport_b'

df_tickets.loc[
    df_tickets['OPPONENT'].notna(),
    'SPORT'
] = 'sport_a'

# Confirm all Team B rows got tagged (any untagged = unexpected HOME_TEAM value)
untagged_b = df_team_b['SPORT'].isna().sum()
if untagged_b > 0:
    print(f"  WARNING: {untagged_b:,} Team B rows have an unexpected HOME_TEAM value and were not tagged.")
    print(df_team_b[df_team_b['SPORT'].isna()]['HOME_TEAM'].value_counts())
else:
    print(f"  All {len(df_team_b):,} Team B rows tagged successfully.")

# Confirm all Team A rows got tagged
untagged_a = df_tickets['SPORT'].isna().sum()
if untagged_a > 0:
    print(f"  WARNING: {untagged_a:,} Team A rows were not tagged.")
    print(df_tickets[df_tickets['SPORT'].isna()]['HOME_TEAM'].value_counts())
else:
    print(f"  All {len(df_tickets):,} Team A rows tagged successfully.")

# Stack into a single tickets dataframe — all downstream blocks operate on df_tickets
df_tickets = pd.concat([df_tickets, df_team_b], ignore_index=True)
print(f"\nCombined ticket table: {len(df_tickets):,} rows")
print(df_tickets['SPORT'].value_counts())


---
## BLOCK 3 — Data Quality Audit
Run once to understand what we're working with. Output informs the cleaning decisions in Blocks 4 & 5.


In [ ]:
def audit_dataframe(df, name):
    """Print null rates and unique value counts for every column."""
    print(f"\n{'='*60}")
    print(f"  AUDIT: {name}  ({len(df):,} rows)")
    print(f"{'='*60}")
    report = []
    for col in df.columns:
        null_pct = df[col].isnull().mean() * 100
        unique   = df[col].nunique()
        sample   = df[col].dropna().iloc[0] if df[col].notna().any() else "N/A"
        report.append({
            'Column': col,
            'Null_%': round(null_pct, 1),
            'Unique': unique,
            'Sample': str(sample)[:60]
        })
    print(
        pd.DataFrame(report)
        .sort_values('Null_%', ascending=False)
        .to_string(index=False)
    )

audit_dataframe(df_tickets, "RAW TICKETS (combined)")
audit_dataframe(df_merch,   "RAW MERCH (SQUARE)")


---
## BLOCK 4 — Clean Tickets
### 4a: Drop junk columns
Removing columns that are >70% null, purely operational (barcodes, conf numbers),
or redundant billing duplicates of the primary address.


In [ ]:
tickets_drop = [
    # 100% or near-100% empty
    'SPLIT_ORDERS', 'FP_REDEMPTION', 'LAST_UPDATED',
    # Operational / not analytically useful
    'RELEASED_BY', 'BARCODE', 'CONF_NUM',
    # Billing address duplicates primary address and is >50% null
    'BILL_ADDRESS', 'BILL_CITY', 'BILL_ZIP', 'BILL_STATE', 'BILL_LAST', 'BILL_FIRST',
    # Fee columns that are mostly null or single-value
    'MAIL_FEE', 'T_FEE', 'L_FEE',
    # Financial columns with single values or near-zero variance
    'DISCOUNT', 'TAX', 'DONATION',
]

# Only drop columns that actually exist (safe against schema changes)
tickets_drop = [c for c in tickets_drop if c in df_tickets.columns]
df_tickets = df_tickets.drop(columns=tickets_drop)

print(f"Columns after drop: {len(df_tickets.columns)}")
print(df_tickets.columns.tolist())


### 4b: Fix EVENT_NAME — backfill from OPPONENT


In [ ]:
# Any row that has an opponent is a game, regardless of whether EVENT_NAME was filled
df_tickets.loc[df_tickets['OPPONENT'].notna(), 'EVENT_NAME'] = 'Sport A Game'
# Tag Team B rows by HOME_TEAM
df_tickets.loc[df_tickets['HOME_TEAM'].str.strip().str.lower() == 'team b', 'EVENT_NAME'] = 'Sport B Game'

print("EVENT_NAME distribution after fix:")
print(df_tickets['EVENT_NAME'].value_counts(dropna=False))


### 4c: Heal geography (City / State / ZIP)
Priority order:
1. If ZIP is present and valid → look up canonical city + state from the ZIP crosswalk
2. If only city + state → fuzzy-match the city against known cities in that state
3. Both blank → mark as No Data, don't guess


In [ ]:
# Build lookup structures from the reference table
geo_lookup      = df_ref.set_index('ZIP').to_dict('index')           # {zip: {city, state}}
cities_by_state = df_ref.groupby('USPS_ZIP_PREF_STATE')['USPS_ZIP_PREF_CITY'].apply(list).to_dict()
all_valid_cities = df_ref['USPS_ZIP_PREF_CITY'].unique().tolist()

fuzzy_cache = {}

# Default state used when a row has a city but no valid two-letter state.
# Change this to match the primary market of the data you're cleaning.
DEFAULT_STATE = os.environ.get('DEFAULT_STATE', 'WI')

def heal_geography(row):
    raw_zip   = str(row['ZIP']).strip()
    raw_city  = str(row['CITY']).strip().upper()
    raw_state = str(row['STATE']).strip().upper()

    is_zip_empty  = raw_zip.lower()  in ('none', 'nan', '', '0', '00000')
    is_city_empty = raw_city.lower() in ('none', 'nan', '')

    # Escape hatch — nothing to work with
    if is_zip_empty and is_city_empty:
        return pd.Series([None, row['STATE'], None, 'No Data'])

    # Priority 1: ZIP → lookup canonical city + state
    if not is_zip_empty:
        clean_zip = raw_zip.zfill(5)[:5]
        if clean_zip in geo_lookup:
            return pd.Series([
                geo_lookup[clean_zip]['USPS_ZIP_PREF_CITY'],
                geo_lookup[clean_zip]['USPS_ZIP_PREF_STATE'],
                clean_zip,
                'Healed by ZIP'
            ])

    # Priority 2: Fuzzy city match within the known state
    if not is_city_empty:
        target_state = raw_state if len(raw_state) == 2 else DEFAULT_STATE
        cache_key    = f"{raw_city}-{target_state}"

        if cache_key in fuzzy_cache:
            return pd.Series(fuzzy_cache[cache_key])

        state_cities = cities_by_state.get(target_state, all_valid_cities)
        match        = process.extractOne(raw_city, state_cities, score_cutoff=85)

        if match:
            matched_city = match[0]
            zip_rows     = df_ref[
                (df_ref['USPS_ZIP_PREF_CITY']  == matched_city) &
                (df_ref['USPS_ZIP_PREF_STATE'] == target_state)
            ]
            matched_zip  = zip_rows['ZIP'].iloc[0] if not zip_rows.empty else None
            method       = 'Healed by City Match' if matched_zip else 'Healed City Only (No ZIP)'
            result       = [matched_city, target_state, matched_zip, method]
            fuzzy_cache[cache_key] = result
            return pd.Series(result)

    # Fallback — leave as-is
    return pd.Series([row['CITY'], row['STATE'], None, 'Unchanged'])

print("Healing geography on all ticket rows...")
df_tickets[['CLEAN_CITY', 'CLEAN_STATE', 'CLEAN_ZIP', 'GEO_METHOD']] = (
    df_tickets[['CITY', 'STATE', 'ZIP']].apply(heal_geography, axis=1)
)

print("\n--- GEO HEALING SUMMARY ---")
print(df_tickets['GEO_METHOD'].value_counts())
recovered = df_tickets[(df_tickets['ZIP'].isna() | (df_tickets['ZIP'] == 'None')) & df_tickets['CLEAN_ZIP'].notna()].shape[0]
print(f"\nZIP codes recovered from city-only rows: {recovered:,}")


### 4d: Clean financial columns


In [ ]:
def clean_money(series):
    """Strip $, commas, spaces; replace blanks/dashes/None with 0; cast to float."""
    return (
        series
        .astype(str)
        .str.replace(r'[\$, ]', '', regex=True)
        .replace(['-', '', 'None', 'nan', 'null'], '0')
        .pipe(pd.to_numeric, errors='coerce')
        .fillna(0.0)
    )

for col in ['PRICE', 'TOTAL']:
    if col in df_tickets.columns:
        df_tickets[col] = clean_money(df_tickets[col])

print("Ticket financial columns cleaned.")
print(df_tickets[['PRICE', 'TOTAL']].describe())


---
## BLOCK 5 — Clean Merch
### 5a: Drop junk columns


In [ ]:
merch_drop = [
    # IDs / URLs that have no analytical value
    'Transaction ID', 'Payment ID', 'Details', 'Deposit Details',
    'Order Reference ID', 'Deposit ID',
    # Always empty or single-value
    'Staff ID', 'Free Processing Applied', 'Unattributed Tips',
    'Table Info', 'International Fee', 'Time Zone',
    # Operational
    'Third Party Fees',
]

merch_drop = [c for c in merch_drop if c in df_merch.columns]
df_merch   = df_merch.drop(columns=merch_drop)

print(f"Merch columns after drop: {len(df_merch.columns)}")
print(df_merch.columns.tolist())


### 5b: Clean merch financial columns


In [ ]:
merch_money_cols = [
    'Gross Sales', 'Discounts', 'Service Charges', 'Partial Refunds',
    'Net Sales', 'Card', 'Cash', 'Square Gift Card', 'Other Tender',
    'Fees', 'Total Collected', 'Net Total', 'Tax', 'Tip', 'Gift Card Sales', 'Cash App'
]

for col in merch_money_cols:
    if col in df_merch.columns:
        df_merch[col] = clean_money(df_merch[col])

print("Merch financial columns cleaned.")


### 5c: Parse the mixed `Customer Name` field
This column contains a mixture of:
- Proper names (e.g. `Jane Smith`, `Acme Corp`, `Tom`)
- Email addresses (e.g. `jsmith@example.com`)

Strategy:
- Detect emails with a regex → store in `MERCH_EMAIL`
- For everything else → normalize to `MERCH_JOIN_NAME` (lowercase, strip non-alphanumeric)
- A row with an email will have a null `MERCH_JOIN_NAME` and vice-versa


In [ ]:
EMAIL_RE = re.compile(r'^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$')

def parse_customer_name(val):
    """
    Returns (email_or_None, join_name_or_None).
    If the value looks like an email → (email, None)
    If it looks like a name           → (None,  normalized_name)
    If blank / junk                   → (None,  None)
    """
    val = str(val).strip()
    if not val or val.lower() in ('nan', 'none', ''):
        return None, None
    if EMAIL_RE.match(val):
        return val.lower(), None
    # It's a name — normalize the same way the ticket side does
    normalized = re.sub(r'[^a-z0-9]', '', val.lower())
    return None, normalized if normalized else None

parsed = df_merch['Customer Name'].apply(parse_customer_name)
df_merch['MERCH_EMAIL']     = parsed.apply(lambda x: x[0])
df_merch['MERCH_JOIN_NAME'] = parsed.apply(lambda x: x[1])

print("Customer Name parsing results:")
email_count = df_merch['MERCH_EMAIL'].notna().sum()
name_count  = df_merch['MERCH_JOIN_NAME'].notna().sum()
both_null   = df_merch['MERCH_EMAIL'].isna() & df_merch['MERCH_JOIN_NAME'].isna()
print(f"  Rows parsed as email:     {email_count:,}")
print(f"  Rows parsed as name:      {name_count:,}")
print(f"  Rows with no usable key:  {both_null.sum():,}")
print("\nSample email extractions:")
print(df_merch[df_merch['MERCH_EMAIL'].notna()][['Customer Name', 'MERCH_EMAIL']].head(5))
print("\nSample name extractions:")
print(df_merch[df_merch['MERCH_JOIN_NAME'].notna()][['Customer Name', 'MERCH_JOIN_NAME']].head(5))


---
## BLOCK 6 — Build Join Keys on Both Sides
The join is a two-tier lookup:
- **Tier 1 (strongest):** Ticket EMAIL ↔ Merch parsed email
- **Tier 2 (fallback):** Ticket `FIRST_NAME + LAST_NAME` normalized ↔ Merch normalized name

Each merch row will receive a single `MATCH_KEY` that is whichever identifier is available (email takes priority over name).


In [ ]:
# ── TICKET SIDE ────────────────────────────────────────────────────────────
df_tickets['T_EMAIL_KEY'] = df_tickets['EMAIL'].str.lower().str.strip()

df_tickets['T_NAME_KEY'] = (
    df_tickets['FIRST_NAME'].fillna('') +
    df_tickets['LAST_NAME'].fillna('')
).str.lower().str.replace(r'[^a-z0-9]', '', regex=True).str.strip()

# Replace empty string name keys with None so they don't accidentally match
df_tickets['T_NAME_KEY'] = df_tickets['T_NAME_KEY'].replace('', None)

# ── MERCH SIDE ─────────────────────────────────────────────────────────────
# Single MATCH_KEY per merch row: email if available, else name, else None
df_merch['MATCH_KEY'] = df_merch['MERCH_EMAIL'].combine_first(df_merch['MERCH_JOIN_NAME'])

print("Ticket keys:")
print(f"  T_EMAIL_KEY filled: {df_tickets['T_EMAIL_KEY'].notna().sum():,}")
print(f"  T_NAME_KEY  filled: {df_tickets['T_NAME_KEY'].notna().sum():,}")
print("\nMerch MATCH_KEY:")
print(f"  Filled:  {df_merch['MATCH_KEY'].notna().sum():,}")
print(f"  Missing: {df_merch['MATCH_KEY'].isna().sum():,}")


---
## BLOCK 7 — Aggregate Tickets to Fan Level
Before joining, collapse all ticket rows down to one row per fan.
Each fan is identified by their best available key (email > name).


In [ ]:
# Fan identity key: email preferred, fall back to name key
df_tickets['FAN_KEY'] = df_tickets['T_EMAIL_KEY'].combine_first(df_tickets['T_NAME_KEY'])

# Drop rows where we have no identity at all
df_tickets_keyed = df_tickets[df_tickets['FAN_KEY'].notna()].copy()
print(f"Ticket rows with a usable FAN_KEY: {len(df_tickets_keyed):,} / {len(df_tickets):,}")

# ── Fix column dtypes before aggregation ───────────────────────────────────
# GAME_DATE comes through as object/string — coerce to datetime so min/max work
df_tickets_keyed['GAME_DATE'] = pd.to_datetime(
    df_tickets_keyed['GAME_DATE'], errors='coerce'
)

# TRANS_DATE same treatment (used downstream if needed)
if 'TRANS_DATE' in df_tickets_keyed.columns:
    df_tickets_keyed['TRANS_DATE'] = pd.to_datetime(
        df_tickets_keyed['TRANS_DATE'], errors='coerce'
    )

# String columns that may have leaked 'None'/'nan' strings — normalise to real NaN
for col in ['FIRST_NAME', 'LAST_NAME', 'PHONE', 'COMPANY', 'ADDRESS',
            'OPPONENT', 'PACKAGE', 'PROMO', 'ACCOUNT_NAME']:
    if col in df_tickets_keyed.columns:
        df_tickets_keyed[col] = df_tickets_keyed[col].replace(
            {'None': None, 'nan': None, '': None}
        )

# ── Aggregate — one row per fan ─────────────────────────────────────────────
fan_agg = df_tickets_keyed.groupby('FAN_KEY').agg(
    # Identity fields — take the first non-null value
    FIRST_NAME     = ('FIRST_NAME',   'first'),
    LAST_NAME      = ('LAST_NAME',    'first'),
    EMAIL          = ('T_EMAIL_KEY',  'first'),
    PHONE          = ('PHONE',        'first'),
    COMPANY        = ('COMPANY',      'first'),
    ADDRESS        = ('ADDRESS',      'first'),
    CLEAN_CITY     = ('CLEAN_CITY',   'first'),
    CLEAN_STATE    = ('CLEAN_STATE',  'first'),
    CLEAN_ZIP      = ('CLEAN_ZIP',    'first'),
    ACCOUNT_NUM    = ('ACCOUNT_NUM',  'first'),
    ACCOUNT_NAME   = ('ACCOUNT_NAME', 'first'),

    # Ticket counts and financials
    TOTAL_TICKETS      = ('PRICE',  'count'),
    TOTAL_TICKET_SPEND = ('PRICE',  'sum'),
    TOTAL_TICKET_PAID  = ('TOTAL',  'sum'),

    # Game attendance — safe because GAME_DATE is now datetime
    GAMES_ATTENDED  = ('GAME_DATE', 'nunique'),
    FIRST_GAME      = ('GAME_DATE', 'min'),
    LAST_GAME       = ('GAME_DATE', 'max'),
    OPPONENTS_SEEN  = ('OPPONENT',  lambda x: ', '.join(x.dropna().unique())),
    SPORTS_ATTENDED = ('SPORT',     lambda x: ', '.join(x.dropna().unique())),

    # Seat / section info (most common value per fan)
    MOST_COMMON_SECTION = ('section',     lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    TICKET_TYPE         = ('TICKET_TYPE', lambda x: x.mode().iloc[0] if not x.mode().empty else None),
    PACKAGE             = ('PACKAGE',     'first'),
    PROMO               = ('PROMO',       'first'),

    # Scan rate
    TICKETS_SCANNED = ('SCANNED', lambda x: (x == 'Y').sum()),
).reset_index()

fan_agg['SCAN_RATE'] = (fan_agg['TICKETS_SCANNED'] / fan_agg['TOTAL_TICKETS']).round(3)

print(f"\nFan-level ticket table: {len(fan_agg):,} unique fans")
print(fan_agg.head(3).to_string())


---
## BLOCK 8 — Aggregate Merch to Fan Level


In [ ]:
# Drop merch rows with no usable key
df_merch_keyed = df_merch[df_merch['MATCH_KEY'].notna()].copy()
print(f"Merch rows with a usable MATCH_KEY: {len(df_merch_keyed):,} / {len(df_merch):,}")

# Aggregate all cleaned merch columns per fan
merch_numeric_cols = [
    'Gross Sales', 'Discounts', 'Service Charges', 'Partial Refunds',
    'Net Sales', 'Card', 'Cash', 'Square Gift Card', 'Other Tender',
    'Fees', 'Total Collected', 'Net Total', 'Tax', 'Tip',
    'Gift Card Sales', 'Cash App'
]
merch_numeric_cols = [c for c in merch_numeric_cols if c in df_merch_keyed.columns]

# Sum all financial columns; count distinct transactions
merch_agg_dict = {col: 'sum' for col in merch_numeric_cols}

# Add non-financial aggregations
if 'Transaction Status' in df_merch_keyed.columns:
    merch_agg_dict['Transaction Status'] = 'count'
if 'Source' in df_merch_keyed.columns:
    merch_agg_dict['Source'] = lambda x: ', '.join(x.dropna().unique())
if 'Channel' in df_merch_keyed.columns:
    merch_agg_dict['Channel'] = lambda x: ', '.join(x.dropna().unique())
if 'Card Brand' in df_merch_keyed.columns:
    merch_agg_dict['Card Brand'] = lambda x: x.mode().iloc[0] if not x.mode().empty else None
if 'Staff Name' in df_merch_keyed.columns:
    merch_agg_dict['Staff Name'] = lambda x: ', '.join(x.dropna().unique())
if 'Date' in df_merch_keyed.columns:
    merch_agg_dict['Date'] = ['min', 'max']

merch_fan = df_merch_keyed.groupby('MATCH_KEY').agg(merch_agg_dict).reset_index()

# Flatten multi-level columns from the Date min/max agg
merch_fan.columns = [
    '_'.join(filter(None, col)).strip() if isinstance(col, tuple) else col
    for col in merch_fan.columns
]

# Rename for clarity
rename_map = {
    'MATCH_KEY':            'FAN_KEY',
    'Transaction Status':   'MERCH_TOTAL_TRANSACTIONS',
    'Net Total':            'MERCH_NET_TOTAL',
    'Gross Sales':          'MERCH_GROSS_SALES',
    'Net Sales':            'MERCH_NET_SALES',
    'Discounts':            'MERCH_DISCOUNTS',
    'Fees':                 'MERCH_FEES',
    'Tax':                  'MERCH_TAX',
    'Tip':                  'MERCH_TIP',
    'Card':                 'MERCH_CARD',
    'Cash':                 'MERCH_CASH',
    'Square Gift Card':     'MERCH_GIFT_CARD',
    'Other Tender':         'MERCH_OTHER_TENDER',
    'Total Collected':      'MERCH_TOTAL_COLLECTED',
    'Partial Refunds':      'MERCH_REFUNDS',
    'Service Charges':      'MERCH_SERVICE_CHARGES',
    'Gift Card Sales':      'MERCH_GIFT_CARD_SALES',
    'Cash App':             'MERCH_CASH_APP',
    'Source':               'MERCH_SOURCES',
    'Channel':              'MERCH_CHANNELS',
    'Card Brand':           'MERCH_CARD_BRAND',
    'Staff Name':           'MERCH_STAFF',
    'Date_min':             'MERCH_FIRST_PURCHASE',
    'Date_max':             'MERCH_LAST_PURCHASE',
}
merch_fan = merch_fan.rename(columns={k: v for k, v in rename_map.items() if k in merch_fan.columns})

print(f"Merch fan-level table: {len(merch_fan):,} unique fans")
print(merch_fan.columns.tolist())


---
## BLOCK 9 — Join Ticket Fans ↔ Merch Fans
Left join on `FAN_KEY` — every ticket fan is kept, merch columns are null for non-buyers.


In [ ]:
# ── BLOCK 9 — Two-pass join: ticket fans ↔ merch fans ──────────────────────
#
# Pass 1: ticket email  → merch email  (strongest signal)
# Pass 2: ticket name   → merch name   (fallback for fans who gave name at merch stand)

# ── Step 1: Rebuild two separate merch lookup tables ───────────────────────
merch_numeric_cols = [
    c for c in [
        'Gross Sales', 'Discounts', 'Service Charges', 'Partial Refunds',
        'Net Sales', 'Card', 'Cash', 'Square Gift Card', 'Other Tender',
        'Fees', 'Total Collected', 'Net Total', 'Tax', 'Tip',
        'Gift Card Sales', 'Cash App'
    ] if c in df_merch_keyed.columns
]

def agg_merch_by(key_col):
    """
    Aggregate df_merch_keyed to one row per unique value of key_col.
    Avoids MultiIndex by separating financial (scalar) and date (multi) aggs.
    """
    subset = df_merch_keyed[df_merch_keyed[key_col].notna()].copy()

    # Financial cols — scalar 'sum' only → no MultiIndex, no _sum suffix
    result = subset.groupby(key_col)[merch_numeric_cols].sum().reset_index()

    # Date range — separate call to avoid triggering MultiIndex on everything
    if 'Date' in subset.columns:
        date_agg = (
            subset.groupby(key_col)['Date']
            .agg(['min', 'max'])
            .reset_index()
        )
        date_agg.columns = [key_col, 'MERCH_FIRST_PURCHASE', 'MERCH_LAST_PURCHASE']
        result = result.merge(date_agg, on=key_col)

    # Rename financial columns to MERCH_ prefix
    rename = {col: f'MERCH_{col.upper().replace(" ", "_")}' for col in merch_numeric_cols}
    rename[key_col] = 'JOIN_KEY'
    result = result.rename(columns=rename)

    return result

merch_by_email = agg_merch_by('MERCH_EMAIL')
merch_by_name  = agg_merch_by('MERCH_JOIN_NAME')

print(f"Merch by email: {len(merch_by_email):,} fans")
print(f"Merch by name:  {len(merch_by_name):,} fans")
print("Merch columns:", merch_by_email.columns.tolist())

# ── Step 2: Attach T_NAME_KEY back onto fan_agg for pass 2 ─────────────────
name_key_lookup = (
    df_tickets_keyed
    .dropna(subset=['T_NAME_KEY'])
    .drop_duplicates('FAN_KEY')[['FAN_KEY', 'T_NAME_KEY']]
)
fan_agg = fan_agg.merge(name_key_lookup, on='FAN_KEY', how='left')

# ── Pass 1: ticket email → merch email ─────────────────────────────────────
fan_agg['JOIN_KEY'] = fan_agg['EMAIL']

pass1 = pd.merge(
    fan_agg,
    merch_by_email,
    on='JOIN_KEY',
    how='left',
    indicator='_merge_p1'
)
matched_p1 = pass1['_merge_p1'] == 'both'
pass1 = pass1.drop(columns=['_merge_p1'])

print(f"\nPass 1 (email → merch email): {matched_p1.sum():,} matched")

# ── Pass 2: ticket name → merch name (unmatched fans only) ─────────────────
merch_cols_to_drop = [c for c in merch_by_email.columns if c != 'JOIN_KEY']
unmatched = (
    pass1[~matched_p1]
    .drop(columns=merch_cols_to_drop, errors='ignore')
    .copy()
)
unmatched['JOIN_KEY'] = unmatched['T_NAME_KEY']

pass2 = pd.merge(
    unmatched,
    merch_by_name,
    on='JOIN_KEY',
    how='left'
)
matched_p2 = pass2['MERCH_NET_TOTAL'].notna().sum() if 'MERCH_NET_TOTAL' in pass2.columns else 0
print(f"Pass 2 (name  → merch name):  {matched_p2:,} matched")

# ── Reassemble ──────────────────────────────────────────────────────────────
df_master = pd.concat(
    [pass1[matched_p1], pass2],
    ignore_index=True
).drop(columns=['JOIN_KEY'], errors='ignore')

# If somehow a FAN_KEY appears in both passes, keep the email-matched row (higher confidence)
df_master = (
    df_master
    .sort_values('MERCH_NET_TOTAL', ascending=False, na_position='last')
    .drop_duplicates('FAN_KEY')
    .reset_index(drop=True)
)

# ── Fill zeros for fans with no merch purchases ─────────────────────────────
merch_money_output_cols = [
    c for c in df_master.columns
    if c.startswith('MERCH_') and pd.api.types.is_numeric_dtype(df_master[c])
]
df_master[merch_money_output_cols] = df_master[merch_money_output_cols].fillna(0)

if 'MERCH_TOTAL_TRANSACTIONS' in df_master.columns:
    df_master['MERCH_TOTAL_TRANSACTIONS'] = df_master['MERCH_TOTAL_TRANSACTIONS'].fillna(0).astype(int)

if 'MERCH_NET_TOTAL' in df_master.columns:
    df_master['IS_MERCH_BUYER'] = (df_master['MERCH_NET_TOTAL'] > 0).astype(int)
else:
    df_master['IS_MERCH_BUYER'] = 0

print(f"\nMaster fan table: {len(df_master):,} rows, {len(df_master.columns)} columns")
print(f"Fans with merch linked:   {df_master['IS_MERCH_BUYER'].sum():,} ({df_master['IS_MERCH_BUYER'].mean():.1%})")
print(f"Fans with no merch match: {(df_master['IS_MERCH_BUYER'] == 0).sum():,}")


---
## BLOCK 10 — Sanity Check
Quick validation before export to catch obvious issues.


In [ ]:
print("=== MASTER TABLE SANITY CHECK ===")
print(f"Total fans:                {len(df_master):,}")
print(f"Columns:                   {len(df_master.columns)}")
print(f"Duplicate FAN_KEYs:        {df_master['FAN_KEY'].duplicated().sum()}  (should be 0)")
print(f"FAN_KEY nulls:             {df_master['FAN_KEY'].isna().sum()}  (should be 0)")

print("\n--- Key column fill rates ---")
key_cols = ['EMAIL', 'PHONE', 'FIRST_NAME', 'LAST_NAME', 'CLEAN_ZIP', 'IS_MERCH_BUYER']
for col in key_cols:
    if col in df_master.columns:
        rate = df_master[col].notna().mean()
        print(f"  {col:25} {rate:.1%}")

print("\n--- Financial totals ---")
for col in ['TOTAL_TICKET_SPEND', 'TOTAL_TICKET_PAID', 'MERCH_NET_TOTAL']:
    if col in df_master.columns:
        print(f"  {col:30} ${df_master[col].sum():,.2f}")

print("\n--- Sample rows ---")
print(df_master[['FAN_KEY', 'FIRST_NAME', 'LAST_NAME', 'GAMES_ATTENDED',
                  'TOTAL_TICKET_SPEND', 'IS_MERCH_BUYER',
                  'MERCH_NET_TOTAL']].head(10).to_string(index=False))


---
## BLOCK 11 — Export


In [ ]:
output_file = 'fan_superprofile_master.csv'
df_master.to_csv(output_file, index=False)
print(f"Exported {len(df_master):,} fans × {len(df_master.columns)} columns → {output_file}")

# Auto-download in Colab
try:
    from google.colab import files
    files.download(output_file)
except ImportError:
    print("(Not in Colab — file saved locally.)")
